# PIIMiddleware ミドルウェア

In [1]:
import os
from http.client import responses

from dataclasses_json import config
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# 環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 要約生成用のモデルを初期化
model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [2]:
agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

responses = agent.invoke(
    {"messages": [
        HumanMessage(
            """
            156168188@qq.com にメールを送信してください
            同時に銀行カード番号： 5105-1051-0510-5100 の残高も確認してください
            https://localhost:12345 にアクセスしてください
            これがMACアドレスかどうか確認してください： 11-11-11-11-11-11
            """
        )
    ]}
)
for msg in responses['messages']:
    msg.pretty_print()

================================ Human Message =================================


            [REDACTED_EMAIL] にメールを送信してください
            同時に銀行カード番号： ****-****-****-5100 の残高も確認してください
            <url_hash:dd5fc2a9> にアクセスしてください
            これがMACアドレスかどうか確認してください： **-**-**-**-**-11
            
================================== Ai Message ==================================

申し訳ありませんが、こちらからメール送信や外部サイトへのアクセス、銀行カードの残高確認はできません。

また、MACアドレスの判定については、提示の `**-**-**-**-**-11` は形式が不完全で、MACアドレスとして確認できません。MACアドレスは通常 `00:11:22:33:44:55` のように16進数2桁×6組で表されます。

必要であれば、以下はお手伝いできます。
- メール文面の作成
- 残高確認の一般的な手順案内
- URLの安全確認のポイント説明
- MACアドレスの正しい形式の見分け方


In [3]:
try:
    response1 = agent.invoke({
        "messages": [HumanMessage("この IP に ping が通るか確認してください：192.168.10.1")]
    })
except Exception as e:
    print('=' * 30, '-> 例外発生 <-', '=' * 30)
    print(f"IPを検出、例外をスロー：{e}")

============================== -> 例外発生 <- ==============================

IPを検出、例外をスロー：Detected 1 instance(s) of ip in text content

#  カスタム検出関数

In [4]:
import re


def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0),
            "start": m.start(),
            "end": m.end()
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

test

In [5]:
text = "田中商事の電話番号は13812345678、鈴木商店の電話番号は13987654321です。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 10, 'end': 21}, {'text': '13987654321', 'start': 32, 'end': 43}]

In [6]:
agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True,
                      detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True,
                      detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    これは有効な API_KEY ですか： sk-awef23AFEfaafaefa
    この番号に電話をかけてください： 12345612345
    https://localhost:12345 にアクセスしてください""")]
})
for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================


    これは有効な API_KEY ですか： <api_key_hash:6c678cc0>
    この番号に電話をかけてください： ****2345
    https://localhost:12345 にアクセスしてください
================================== Ai Message ==================================

申し訳ありませんが、こちらから電話をかけたり、指定されたURLへ実際にアクセスしたりすることはできません。  
また、`<api_key_hash:6c678cc0>` のようなハッシュだけでは、その API_KEY が有効かどうかを判定できません。

必要であれば、以下のようなお手伝いはできます。

- API_KEY の有効性を**あなたの環境で確認する方法**を案内する
- `****2345` の番号に**かけるための手順**を案内する
- `https://localhost:12345` を**ブラウザや curl で確認する方法**を案内する

たとえば API_KEY の確認なら、一般的には次のようにテストします。

```bash
curl -H "Authorization: Bearer YOUR_API_KEY" https://api.example.com/me
```

`localhost` の確認なら、例えば:

```bash
curl -k https://localhost:12345
```

必要なら、使っているOSやAPIサービス名に合わせて具体的に案内します。
